# Synthetic Datasets: Plots of Utility and Group Fairness Metrics

Author: Ilse Harmers \
Last modified: March 26, 2026

In [ ]:
# Importing libraries.
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
from matplotlib import ticker
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines

import matplotlib
params = {'axes.titlesize':'20',
          'xtick.labelsize':'18',
          'ytick.labelsize':'18',
          'font.size':'18',
          'legend.fontsize':'medium',
          'lines.linewidth':'2.0',
          'font.weight':'normal',
          'lines.markersize':'5',
          'text.latex.preamble': r'\usepackage{amsfonts}',
          }
matplotlib.rcParams.update(params)
plt.rcParams["mathtext.fontset"] = "cm"
plt.rc('text', usetex=True)
plt.rc('font', family='serif')

## Initialization

In [ ]:
# Initializing the values for epsilon, the labels for the different metrics, the weights used in the guidelines and our functions. 
epsi = [1, 2, 5, 8]
metrics = ["acc", "prec", "recall", "auroc", "dem", "dis", "eqopp"]
weights = [1, 1, 0]   # 1: auroc; 1: group fairness; 0: privacy

def synth_selection(models, epsilon, dataset, weights = [1, 1, 1]):
    """This function returns the rankings of the 'best' synthetic datasets out of all our models for an epsilon 'epsilon'. 'Best' refers to the 
    datasets which are best able to balance classifier utility (AUROC), classifier group fairness (demographic parity, disparate impact and equal 
    opportunity) and privacy (DCR and NNDR). The rankings are based on a point-based ranking scheme where the 'best' results for each of the metrics 
    are ranked from first through last place. This function also returns the (un)processed utility and fairness results and the privacy results
    of the synthetic datasets. 

    The total number of synthetic datasets for each model is hard-coded as 15, which should be modified in cases where this is not true.
    
    models [list]: folder locations of the GANs' synthetic datasets
    epsilon [integer]: epsilon value 
    dataset [string]: folder name of the synthetic datasets; can be set to "adult", "bank-marketing" or "credit-card-default"
    weights [list]: utility, fairness and privacy weights assigned to the AUROC, classifier group fairness and privacy points of the synthetic datasets
    """
    
    metrics = ["acc", "auroc", "dem", "dis", "eqopp"]
    results_eps_avg = {}
    results_eps_avg_std = {}
    privacy_results = {}
    
    for mod in models:
        if mod == "DP-GAN" or mod == "FairDP-GAN(dis)":
            file_path = f"../synthetic-datasets_{mod}_B=512/{dataset}/"
        else:
            file_path = f"./synthetic-datasets_{mod}/{dataset}/"

        # Importing synthetic results as pandas DataFrames.
        run1 = pd.read_csv(file_path + f"epsi_{epsilon}/run1/results_epsi-{epsilon}_run1.csv", index_col = 0, na_values = "np.nan")
        run2 = pd.read_csv(file_path + f"epsi_{epsilon}/run2/results_epsi-{epsilon}_run2.csv", index_col = 0, na_values = "np.nan")
        run3 = pd.read_csv(file_path + f"epsi_{epsilon}/run3/results_epsi-{epsilon}_run3.csv", index_col = 0, na_values = "np.nan")
        run4 = pd.read_csv(file_path + f"epsi_{epsilon}/run4/results_epsi-{epsilon}_run4.csv", index_col = 0, na_values = "np.nan")
        run5 = pd.read_csv(file_path + f"epsi_{epsilon}/run5/results_epsi-{epsilon}_run5.csv", index_col = 0, na_values = "np.nan")
        
        # Concatenating the results into one DataFrame. 
        results = pd.concat([run1.reset_index(drop = True), run2.reset_index(drop = True),
                             run3.reset_index(drop = True), run4.reset_index(drop = True),
                             run5.reset_index(drop = True)], axis = 0)

        eps_avg = {}
        eps_avg_std = {}

        dem_mean = np.nanmean(abs(results[["dem-parity"]]), axis = 1)
        eps_avg["dem-parity"] = {f"sample{i + 1}-{mod}": dem_mean[i] for i in range(len(dem_mean))}
        eps_avg_std["dem-parity"] = {f"sample{i + 1}-{mod}": "NaN" for i in range(len(dem_mean))}
        
        dis_mean = np.nanmean(results[["dis-impact"]], axis = 1)
        eps_avg["dis-impact"] = {f"sample{i + 1}-{mod}": dis_mean[i] for i in range(len(dis_mean))}
        eps_avg_std["dis-impact"] = {f"sample{i + 1}-{mod}": "NaN" for i in range(len(dis_mean))}
            
        for m in metrics:
            if m == "dis":   # averaging over 'regular' values
                m_mean = np.nanmean(results[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]], axis = 1)
                m_std = np.nanstd(results[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]], axis = 1)
            else:   # averaging over absolute values
                m_mean = np.nanmean(abs(results[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]]), axis = 1)
                m_std = np.nanstd(abs(results[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]]), axis = 1)
            eps_avg[m] = {f"sample{i + 1}-{mod}": m_mean[i] for i in range(len(m_mean))}
            eps_avg_std[m] = {f"sample{i + 1}-{mod}": m_std[i] for i in range(len(m_std))}

        results_eps_avg[mod] = pd.DataFrame(eps_avg)
        results_eps_avg_std[mod] = pd.DataFrame(eps_avg_std)

        new_index = [f"sample{i + 1}-{mod}" for i in range(len(m_mean))]
        if mod == "TabFair":
            privacy_results[mod] = pd.read_csv(file_path + f"priv-results.csv", index_col = 0).T
        else:
            privacy_results[mod] = pd.read_csv(file_path + f"priv-results_epsi-{epsilon}.csv", index_col = 0).T
        privacy_results[mod].rename(index = {f"sample{i + 1}": f"sample{i + 1}-{mod}" for i in range(len(m_mean))}, inplace = True) 

    results_avg = pd.concat([results_eps_avg[k] for k in results_eps_avg.keys()], axis = 0)
    results_avg_std = pd.concat([results_eps_avg_std[k] for k in results_eps_avg_std.keys()], axis = 0)
    priv_results = pd.concat([privacy_results[k] for k in privacy_results.keys()], axis = 0)
    
    # Rankings for classfiers' utility.
    points_auroc = pd.Series(range(15*len(models), 0, -1), index = results_avg.sort_values(by = ["auroc"]).index)

    # Rankings for classfiers' group fairness.
    points_dem = pd.Series(range(1, 15*len(models) + 1), index = results_avg.sort_values(by = ["dem"]).index)
    points_eqopp = pd.Series(range(1, 15*len(models) + 1), index = results_avg.sort_values(by = ["eqopp"]).index)
    # We set all DI values in the fair/green zone (i.e., between 0.0 - 0.45) to 1 to single these out, assuming that no DI value is exactly equal to 1.
    points_dis = results_avg["dis"].copy()
    points_dis.loc[(points_dis >= 0) & (points_dis <= 0.45)] = 1
    # For DI values in the red zones (i.e., smaller than 0 or greater than 0.45), we compute their distance to the green zone. 
    # Smaller distances to the green zone are fairer. 
    points_dis.loc[(points_dis > 0.45) & (points_dis != 1.0)] = points_dis - 0.45
    points_dis.loc[points_dis < 0] = abs(points_dis)
    points_dis[points_dis.loc[points_dis != 1.0].sort_values().index] = range(2, 2 + len(points_dis.loc[points_dis != 1.0].sort_values()))
    points_gpfair = (points_dem + points_dis + points_eqopp) / 3

    # Rankings for synthetic datasets' privacy.
    points_NNDR = pd.Series(range(15*len(models), 0, -1), index = priv_results.sort_values(by = ["NNDR"]).index)
    points_DCR = pd.Series(range(15*len(models), 0, -1), index = priv_results.sort_values(by = ["DCR"]).index)
    points_priv = (points_NNDR + points_DCR) / 2

    final_srt = (weights[0]*points_auroc + weights[1]*points_gpfair + weights[2]*points_priv).sort_values()

    return (results, results_avg, results_avg_std, privacy_results, final_srt)

def results(model, epsilons, dataset, weights):
    """This function returns the 'best' synthetic dataset out of all our runs for each epsilon value in 'epsilons'. 'Best' refers to the dataset which 
    is best able to balance classifier utility (AUROC), classifier group fairness (demographic parity, disparate impact and equal opportunity) and 
    privacy (DCR and NNDR). This function also returns the (un)processed utility and fairness results and the privacy results of the 'best' 
    synthetic datasets.

    model [list]: model name of GAN, recorded in a single-entry list
    epsilons [list]: epsilon values during GAN training; if GAN has no privacy constraints, set epsilons = [0]
    dataset [string]: folder name of the synthetic datasets; can be set to "adult", "bank-marketing" or "credit-card-default"
    weights [list]: utility, fairness and privacy weights assigned to the AUROC, classifier group fairness and privacy points of the synthetic datasets
    """
    
    results_mean = {}
    results_std = {}
    results_org = {}
    results_priv = {}
    for e in epsilons:
        (results, results_avg, results_avg_std, privacy_results, final_srt) = synth_selection(model, epsilon = e, dataset = dataset, weights = weights)
        if e != 0:
            print(f"Epsilon = {e}")
        try:
            index = int(final_srt.index[0][6:8]) - 1
        except  ValueError:
            index = int(final_srt.index[0][6:7]) - 1
        print(f"Sample {index + 1}")   # best synthetic dataset
        # Privacy metrics of the best synthetic dataset.
        print("DCR:", privacy_results[model[0]].loc[final_srt.index[0]][0])
        print("NNDR:", privacy_results[model[0]].loc[final_srt.index[0]][1])
        print()
        # Dataset utility metrics of the best synthetic dataset.
        if model[0] == "TabFair":
            file_path = f"./synthetic-datasets_{model[0]}/{dataset}/"
            dat_util = pd.read_csv(file_path + "dataset-utility-results_mean.csv", index_col = 0)
            dat_util_std = pd.read_csv(file_path + "dataset-utility-results_std.csv", index_col = 0)
        elif model[0] == "DP-GAN" or model[0] == "FairDP-GAN(dis)":
            file_path = f"../synthetic-datasets_{model[0]}_B=512/{dataset}/"
            dat_util = pd.read_csv(file_path + f"dataset-utility-results_mean_epsi-{e}.csv", index_col = 0)
            dat_util_std = pd.read_csv(file_path + f"dataset-utility-results_std_epsi-{e}.csv", index_col = 0)
        else:
            file_path = f"./synthetic-datasets_{model[0]}/{dataset}/"
            dat_util = pd.read_csv(file_path + f"dataset-utility-results_mean_epsi-{e}.csv", index_col = 0)
            dat_util_std = pd.read_csv(file_path + f"dataset-utility-results_std_epsi-{e}.csv", index_col = 0)
        print(dat_util.iloc[index])
        print(dat_util_std.iloc[index])
        if e != 0 and e < 8:
            print()
        if e == 0:
            return (pd.concat([results_avg.loc[final_srt.index[0]], results_avg_std.loc[final_srt.index[0]]], axis = 1, 
                             ignore_index = True).T.rename({0: "mean", 1: "std"}), pd.DataFrame(results.iloc[index]).T,
            pd.DataFrame({"DCR": [privacy_results[model[0]].loc[final_srt.index[0]][0]], 
                          "NNDR": [privacy_results[model[0]].loc[final_srt.index[0]][1]]}))
        else:      
            results_mean[f"epsi_{e}"] = results_avg.loc[final_srt.index[0]]
            results_std[f"epsi_{e}"] = results_avg_std.loc[final_srt.index[0]]
            results_org[f"epsi_{e}"] = results.iloc[index]
            results_priv[f"epsi_{e}"] = {"DCR": privacy_results[model[0]].loc[final_srt.index[0]][0], 
                                         "NNDR": privacy_results[model[0]].loc[final_srt.index[0]][1]}

    results_mean = pd.concat([results_mean[i] for i in [f"epsi_{e}" for e in epsilons]], axis = 1, 
                             ignore_index = True).T.rename(index = {i: f"epsi_{epsilons[i]}" for i in range(len(epsilons))})
    results_std = pd.concat([results_std[i] for i in [f"epsi_{e}" for e in epsilons]], axis = 1, 
                             ignore_index = True).T.rename(index = {i: f"epsi_{epsilons[i]}" for i in range(len(epsilons))})
    results_org = pd.concat([results_org[i] for i in [f"epsi_{e}" for e in epsilons]], axis = 1, 
                             ignore_index = True).T.rename(index = {i: f"epsi_{epsilons[i]}" for i in range(len(epsilons))})
    results_priv = pd.DataFrame(results_priv)
    
    return results_mean, results_std, results_org, results_priv

def metrics_average(adult_samp, bank_samp, credit_samp, epsi = epsi):
    """This function computes the average classifier utility and group fairness results of the Adult, Bank and Credit 'best' synthetic datasets for
    each epsilon value. Both the means and standard deviations of the averaged results are returned."""
    
    avg_met_results = {}
    avg_met_results_std = {}
    for e in range(len(epsi)):
        eps_avg = {}
        eps_avg_std = {}
        for m in metrics:
            if m != "dis":   # averaging over absolute values
                avg = [abs(adult_samp.iloc[e][[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]]), abs(bank_samp.iloc[e][[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]]),
                       abs(credit_samp.iloc[e][[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]])]
            else:   # averaging over 'regular' values
                avg = [adult_samp.iloc[e][[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]], bank_samp.iloc[e][[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]],
                       credit_samp.iloc[e][[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]]]
            eps_avg[m] = np.nanmean(avg)
            eps_avg_std[m] = np.nanstd(avg)
        avg_met_results[f"epsi_{epsi[e]}"] = eps_avg 
        avg_met_results_std[f"epsi_{epsi[e]}"] = eps_avg_std

    avg_met_results = pd.DataFrame(avg_met_results).T
    avg_met_results_std = pd.DataFrame(avg_met_results_std).T

    return avg_met_results, avg_met_results_std

## Adult Dataset (Real)

In [ ]:
adult_results_real = pd.read_csv("./train-test-datasets/adult/results_real.csv", index_col = 0)

results_real_avg = {}
for m in metrics:
    if m == "dis":   # averaging over 'regular' values
        m_mean = adult_results_real[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]].mean().mean()
        m_std = adult_results_real[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]].mean().std(ddof = 0)
    else:   # averaging over absolute values
        m_mean = abs(adult_results_real[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]]).mean().mean()
        m_std = abs(adult_results_real[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]]).mean().std(ddof = 0)
    results_real_avg[m] = (m_mean, m_std)

adult_results = pd.concat([adult_results_real[["dem-parity", "dis-impact"]].reset_index(drop = True),
                           pd.DataFrame(results_real_avg).reset_index(drop = True)], 
                           axis = 1).rename(index = {0: "mean", 1: "std"})

In [ ]:
adult_results

## Synthetic Adult Dataset (DP-GAN)

In [ ]:
# DP-GAN.
model = ["DP-GAN"]
adult_results_DP, adult_results_DP_std, adult_DP_samples, DP_priv_adult = results(model = model, epsilons = epsi, dataset = "adult", weights = weights)

In [ ]:
adult_results_DP#, adult_results_DP_std

## Synthetic Adult Dataset (*Dis* with $\lambda = 0.5$)

In [ ]:
# Fair DP-GAN (dis).
model = ["FairDP-GAN(dis)"]
adult_results_dis, adult_results_dis_std, adult_dis_samples, dis_priv_adult = results(model = model, epsilons = epsi, dataset = "adult",
                                                                                      weights = weights)

In [ ]:
adult_dis_old = pd.read_csv("../saved-GAN-results/adult/adult_results_dis.csv", index_col = 0)
adult_dis_old_std = pd.read_csv("../saved-GAN-results/adult/adult_results_dis_std.csv", index_col = 0)
adult_dis_old

In [ ]:
adult_results_dis#, adult_results_dis_std

## Synthetic Adult Dataset (*Dis* with $\lambda = 0.75$)

In [ ]:
# Fair DP-GAN (dis); lambda = 0.75.
model = ["FairDP-GAN(dis)_l=0.75"]
adult_results_075, adult_results_075_std, adult_075_samples, dis075_priv_adult = results(model = model, epsilons = epsi, dataset = "adult",
                                                                                         weights = weights)

In [ ]:
adult_results_075#, adult_results_075_std

## Synthetic Adult Dataset (*Dis* with $\lambda = 1.0$)

In [ ]:
# Fair DP-GAN (dis); lambda = 1.0.
model = ["FairDP-GAN(dis)_l=1.0"]
adult_results_100, adult_results_100_std, adult_100_samples, dis100_priv_adult = results(model = model, epsilons = epsi, dataset = "adult",
                                                                                         weights = weights)

In [ ]:
adult_results_100#, adult_results_100_std

## Synthetic Adult Dataset (*Dis* with $\lambda = 1.25$)

In [ ]:
# Fair DP-GAN (dis); lambda = 1.25.
model = ["FairDP-GAN(dis)_l=1.25"]
adult_results_125, adult_results_125_std, adult_125_samples, dis125_priv_adult = results(model = model, epsilons = epsi, dataset = "adult",
                                                                                         weights = weights)

In [ ]:
adult_results_125#, adult_results_125_std

## Synthetic Adult Dataset (*Dis* with $\lambda = 1.5$)

In [ ]:
# Fair DP-GAN (dis); lambda = 1.5.
model = ["FairDP-GAN(dis)_l=1.5"]
adult_results_150, adult_results_150_std, adult_150_samples, dis150_priv_adult = results(model = model, epsilons = epsi, dataset = "adult",
                                                                                         weights = weights)

In [ ]:
adult_results_150#, adult_results_150_std

## Bank Marketing Dataset (Real)

In [ ]:
bank_results_real = pd.read_csv("./train-test-datasets/bank-marketing/results_real.csv", index_col = 0)

results_real_avg = {}
for m in metrics:
    if m == "dis":   # averaging over 'regular' values
        m_mean = bank_results_real[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]].mean().mean()
        m_std = bank_results_real[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]].mean().std(ddof = 0)
    else:   # averaging over absolute values
        m_mean = abs(bank_results_real[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]]).mean().mean()
        m_std = abs(bank_results_real[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]]).mean().std(ddof = 0)
    results_real_avg[m] = (m_mean, m_std)

bank_results = pd.concat([bank_results_real[["dem-parity", "dis-impact"]].reset_index(drop = True),
                          pd.DataFrame(results_real_avg).reset_index(drop = True)], 
                          axis = 1).rename(index = {0: "mean", 1: "std"})

In [ ]:
bank_results

## Synthetic Bank Marketing Dataset (DP-GAN)

In [ ]:
# DP-GAN.
model = ["DP-GAN"]
bank_results_DP, bank_results_DP_std, bank_DP_samples, DP_priv_bank = results(model = model, epsilons = epsi, dataset = "bank-marketing",
                                                                              weights = weights)

In [ ]:
bank_DP_old = pd.read_csv("../saved-GAN-results/bank-marketing/bank_results_DP.csv", index_col = 0)
bank_DP_old_std = pd.read_csv("../saved-GAN-results/bank-marketing/bank_results_DP_std.csv", index_col = 0)
bank_DP_old

In [ ]:
bank_results_DP#, bank_results_DP_std

## Synthetic Bank Marketing Dataset (*Dis* with $\lambda = 0.5$)

In [ ]:
# Fair DP-GAN (dis).
model = ["FairDP-GAN(dis)"]
bank_results_dis, bank_results_dis_std, bank_dis_samples, dis_priv_bank = results(model = model, epsilons = epsi, dataset = "bank-marketing",
                                                                                  weights = weights)

In [ ]:
bank_dis_old = pd.read_csv("../saved-GAN-results/bank-marketing/bank_results_dis.csv", index_col = 0)
bank_dis_old_std = pd.read_csv("../saved-GAN-results/bank-marketing/bank_results_dis_std.csv", index_col = 0)
bank_dis_old

In [ ]:
bank_results_dis#, bank_results_dis_std

## Synthetic Bank Marketing Dataset (*Dis* with $\lambda = 0.75$)

In [ ]:
# Fair DP-GAN (dis); lambda = 0.75.
model = ["FairDP-GAN(dis)_l=0.75"]
bank_results_075, bank_results_075_std, bank_075_samples, dis075_priv_bank = results(model = model, epsilons = epsi, dataset = "bank-marketing",
                                                                                     weights = weights)

In [ ]:
bank_results_075#, bank_results_075_std

## Synthetic Bank Marketing Dataset (*Dis* with $\lambda = 1.0$)

In [ ]:
# Fair DP-GAN (dis); lambda = 1.0.
model = ["FairDP-GAN(dis)_l=1.0"]
bank_results_100, bank_results_100_std, bank_100_samples, dis100_priv_bank = results(model = model, epsilons = epsi, dataset = "bank-marketing",
                                                                                     weights = weights)

In [ ]:
bank_results_100#, bank_results_100_std

## Synthetic Bank Marketing Dataset (*Dis* with $\lambda = 1.25$)

In [ ]:
# Fair DP-GAN (dis); lambda = 1.25.
model = ["FairDP-GAN(dis)_l=1.25"]
bank_results_125, bank_results_125_std, bank_125_samples, dis125_priv_bank = results(model = model, epsilons = epsi, dataset = "bank-marketing",
                                                                                     weights = weights)

In [ ]:
bank_results_125#, bank_results_125_std

## Synthetic Bank Marketing Dataset (*Dis* with $\lambda = 1.5$)

In [ ]:
# Fair DP-GAN (dis); lambda = 1.5.
model = ["FairDP-GAN(dis)_l=1.5"]
bank_results_150, bank_results_150_std, bank_150_samples, dis150_priv_bank = results(model = model, epsilons = epsi, dataset = "bank-marketing",
                                                                                     weights = weights)

In [ ]:
bank_results_150#, bank_results_150_std

## Credit Card Default Dataset (Real)

In [ ]:
credit_results_real = pd.read_csv("./train-test-datasets/credit-card-default/results_real.csv", index_col = 0)

results_real_avg = {}
for m in metrics:
    if m == "dis":   # averaging over 'regular' values
        m_mean = credit_results_real[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]].mean().mean()
        m_std = credit_results_real[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]].mean().std(ddof = 0)
    else:   # averaging over absolute values
        m_mean = abs(credit_results_real[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]]).mean().mean()
        m_std = abs(credit_results_real[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]]).mean().std(ddof = 0)
    results_real_avg[m] = (m_mean, m_std)

credit_results = pd.concat([credit_results_real[["dem-parity", "dis-impact"]].reset_index(drop = True),
                            pd.DataFrame(results_real_avg).reset_index(drop = True)], 
                            axis = 1).rename(index = {0: "mean", 1: "std"})

In [ ]:
credit_results

## Synthetic Credit Card Default Dataset (DP-GAN)

In [ ]:
# DP-GAN.
model = ["DP-GAN"]
credit_results_DP, credit_results_DP_std, credit_DP_samples, DP_priv_cred = results(model = model, epsilons = epsi, dataset = "credit-card-default", 
                                                                      weights = weights)

In [ ]:
credit_results_DP#, credit_results_DP_std

## Synthetic Credit Card Default Dataset (*Dis* with $\lambda = 0.5$)

In [ ]:
# Fair DP-GAN (dis).
model = ["FairDP-GAN(dis)"]
credit_results_dis, credit_results_dis_std, credit_dis_samples, dis_priv_cred = results(model = model, epsilons = epsi, 
                                                                                        dataset = "credit-card-default", weights = weights)

In [ ]:
credit_dis_old = pd.read_csv("../saved-GAN-results/credit-card-default/credit_results_dis.csv", index_col = 0)
credit_dis_old_std = pd.read_csv("../saved-GAN-results/credit-card-default/credit_results_dis_std.csv", index_col = 0)
credit_dis_old

In [ ]:
credit_results_dis#, credit_results_dis_std

## Synthetic Credit Card Default Dataset (*Dis* with $\lambda = 0.75$)

In [ ]:
# Fair DP-GAN (dis); lambda = 0.75.
model = ["FairDP-GAN(dis)_l=0.75"]
credit_results_075, credit_results_075_std, credit_075_samples, dis075_priv_cred = results(model = model, epsilons = epsi, 
                                                                                           dataset = "credit-card-default", weights = weights)

In [ ]:
credit_results_075#, credit_results_075_std

## Synthetic Credit Card Default Dataset (*Dis* with $\lambda = 1.0$)

In [ ]:
# Fair DP-GAN (dis); lambda = 1.0.
model = ["FairDP-GAN(dis)_l=1.0"]
credit_results_100, credit_results_100_std, credit_100_samples, dis100_priv_cred = results(model = model, epsilons = epsi, 
                                                                                           dataset = "credit-card-default", weights = weights)

In [ ]:
credit_results_100#, credit_results_100_std

## Synthetic Credit Card Default Dataset (*Dis* with $\lambda = 1.25$)

In [ ]:
# Fair DP-GAN (dis); lambda = 1.25.
model = ["FairDP-GAN(dis)_l=1.25"]
credit_results_125, credit_results_125_std, credit_125_samples, dis125_priv_cred = results(model = model, epsilons = epsi, 
                                                                                           dataset = "credit-card-default", weights = weights)

In [ ]:
credit_results_125#, credit_results_125_std

## Synthetic Credit Card Default Dataset (*Dis* with $\lambda = 1.5$)

In [ ]:
# Fair DP-GAN (dis); lambda = 1.5.
model = ["FairDP-GAN(dis)_l=1.5"]
credit_results_150, credit_results_150_std, credit_150_samples, dis150_priv_cred = results(model = model, epsilons = epsi, 
                                                                                           dataset = "credit-card-default", weights = weights)

In [ ]:
credit_results_150#, credit_results_150_std

## Average of Results

In [ ]:
# Average of the Adult, Bank & Credit datasets' demographic parity for each model. 
avg_dem_real = [adult_results["dem-parity"][0], bank_results["dem-parity"][0], credit_results["dem-parity"][0]]

avg_dem_DP = [[adult_results_DP["dem-parity"][e], bank_results_DP["dem-parity"][e], credit_results_DP["dem-parity"][e]] for e in range(len(epsi))]

avg_dem_fdis = [[adult_results_dis["dem-parity"][e], bank_results_dis["dem-parity"][e], credit_results_dis["dem-parity"][e]] for e in range(len(epsi))]
avg_dem_f075 = [[adult_results_075["dem-parity"][e], bank_results_075["dem-parity"][e], credit_results_075["dem-parity"][e]] for e in range(len(epsi))]
avg_dem_f100 = [[adult_results_100["dem-parity"][e], bank_results_100["dem-parity"][e], credit_results_100["dem-parity"][e]] for e in range(len(epsi))]
avg_dem_f125 = [[adult_results_125["dem-parity"][e], bank_results_125["dem-parity"][e], credit_results_125["dem-parity"][e]] for e in range(len(epsi))]
avg_dem_f150 = [[adult_results_150["dem-parity"][e], bank_results_150["dem-parity"][e], credit_results_150["dem-parity"][e]] for e in range(len(epsi))]

# Average of the Adult, Bank & Credit datasets' disparate impact for each model. 
avg_dis_real = [adult_results["dis-impact"][0], bank_results["dis-impact"][0], credit_results["dis-impact"][0]]

avg_dis_DP = [[adult_results_DP["dis-impact"][e], bank_results_DP["dis-impact"][e], credit_results_DP["dis-impact"][e]] for e in range(len(epsi))]

avg_dis_fdis = [[adult_results_dis["dis-impact"][e], bank_results_dis["dis-impact"][e], credit_results_dis["dis-impact"][e]] for e in range(len(epsi))]
avg_dis_f075 = [[adult_results_075["dis-impact"][e], bank_results_075["dis-impact"][e], credit_results_075["dis-impact"][e]] for e in range(len(epsi))]
avg_dis_f100 = [[adult_results_100["dis-impact"][e], bank_results_100["dis-impact"][e], credit_results_100["dis-impact"][e]] for e in range(len(epsi))]
avg_dis_f125 = [[adult_results_125["dis-impact"][e], bank_results_125["dis-impact"][e], credit_results_125["dis-impact"][e]] for e in range(len(epsi))]
avg_dis_f150 = [[adult_results_150["dis-impact"][e], bank_results_150["dis-impact"][e], credit_results_150["dis-impact"][e]] for e in range(len(epsi))]

In [ ]:
# Average of the datasets' utility and fairness metrics (w.r.t. the trained classifiers).
metrics = ["acc", "prec", "recall", "auroc", "dem", "dis", "eqopp"]

avg_real_results = {}
for m in metrics:
    if m != "dis":   # averaging over absolute values
        avg = [abs(adult_results_real[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]]), abs(bank_results_real[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]]),
               abs(credit_results_real[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]])]
    else:   # averaging over 'regular' values
        avg = [adult_results_real[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]], bank_results_real[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]],
               credit_results_real[[f"rf-{m}", f"mlp-{m}", f"lda-{m}"]]]
    avg_real_results[m] = (np.mean(avg), np.std(avg))
avg_real_results = pd.DataFrame(avg_real_results).rename(index = {0: "mean", 1: "std"})

avg_DP_results, avg_DP_results_std = metrics_average(adult_samp = adult_DP_samples, bank_samp = bank_DP_samples, credit_samp = credit_DP_samples)

avg_fdis_results, avg_fdis_results_std = metrics_average(adult_samp = adult_dis_samples, bank_samp = bank_dis_samples, credit_samp = credit_dis_samples)
avg_f075_results, avg_f075_results_std = metrics_average(adult_samp = adult_075_samples, bank_samp = bank_075_samples, credit_samp = credit_075_samples)
avg_f100_results, avg_f100_results_std = metrics_average(adult_samp = adult_100_samples, bank_samp = bank_100_samples, credit_samp = credit_100_samples)
avg_f125_results, avg_f125_results_std = metrics_average(adult_samp = adult_125_samples, bank_samp = bank_125_samples, credit_samp = credit_125_samples)
avg_f150_results, avg_f150_results_std = metrics_average(adult_samp = adult_150_samples, bank_samp = bank_150_samples, credit_samp = credit_150_samples)

## Plots of Results

In [ ]:
# Plotting dataset fairness metrics of the 'best' synthetic datasets from all our models.
labels = ["Original", "DP-GAN", "$Dis$ ($\lambda_f = 0.5$)", "$Dis$ ($\lambda_f = 0.75$)", "$Dis$ ($\lambda_f = 1.0$)", "$Dis$ ($\lambda_f = 1.25$)",
          "$Dis$ ($\lambda_f = 1.5$)"]
colors = ["red", "purple", "darkcyan", "darkorange", "olive", "magenta", "royalblue"]
limits = {"dem-parity": (-0.03, 0.86), "dis-impact": (-1.05, 1.05)}
fairness_clf = ["dem-parity", "dis-impact"]
fairness_labels = ["Demographic Parity", "Disparate Impact"]
markers = ["o", "D", "P", "^", "s", "v"]
msize = 9

for f in fairness_clf:
    # Create a figure for the aggregated plot.
    fig, axes = plt.subplots(1, 4, figsize = (20, 8.0))  # 1 x 4 grid
    for c in range(0, 4):
        if c == 0:
            dataset = "Adult"
            results_np = pd.concat([adult_results[f]], axis = 1).set_axis(["Real"], axis = 1)
            
            results_DP = pd.concat([adult_results_DP[f], adult_results_DP_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results = pd.concat([results_np, results_DP], axis = 1)

            results_fdisDP = pd.concat([adult_results_dis[f], adult_results_dis_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_fdis = pd.concat([results_np, results_fdisDP], axis = 1)

            results_f075DP = pd.concat([adult_results_075[f], adult_results_075_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_f075 = pd.concat([results_np, results_f075DP], axis = 1)
            results_f100DP = pd.concat([adult_results_100[f], adult_results_100_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_f100 = pd.concat([results_np, results_f100DP], axis = 1)
            results_f125DP = pd.concat([adult_results_125[f], adult_results_125_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_f125 = pd.concat([results_np, results_f125DP], axis = 1)
            results_f150DP = pd.concat([adult_results_150[f], adult_results_150_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_f150 = pd.concat([results_np, results_f150DP], axis = 1)

        elif c == 1:
            dataset = "Bank"
            results_np = pd.concat([bank_results[f]], axis = 1).set_axis(["Real"], axis = 1)
            
            results_DP = pd.concat([bank_results_DP[f], bank_results_DP_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results = pd.concat([results_np, results_DP], axis = 1)

            results_fdisDP = pd.concat([bank_results_dis[f], bank_results_dis_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_fdis = pd.concat([results_np, results_fdisDP], axis = 1)

            results_f075DP = pd.concat([bank_results_075[f], bank_results_075_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_f075 = pd.concat([results_np, results_f075DP], axis = 1)
            results_f100DP = pd.concat([bank_results_100[f], bank_results_100_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_f100 = pd.concat([results_np, results_f100DP], axis = 1)
            results_f125DP = pd.concat([bank_results_125[f], bank_results_125_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_f125 = pd.concat([results_np, results_f125DP], axis = 1)
            results_f150DP = pd.concat([bank_results_150[f], bank_results_150_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_f150 = pd.concat([results_np, results_f150DP], axis = 1)

        elif c == 2:
            dataset = "Credit"
            results_np = pd.concat([credit_results[f]], axis = 1).set_axis(["Real"], axis = 1)
            
            results_DP = pd.concat([credit_results_DP[f], credit_results_DP_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results = pd.concat([results_np, results_DP], axis = 1)

            results_fdisDP = pd.concat([credit_results_dis[f], credit_results_dis_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_fdis = pd.concat([results_np, results_fdisDP], axis = 1)

            results_f075DP = pd.concat([credit_results_075[f], credit_results_075_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_f075 = pd.concat([results_np, results_f075DP], axis = 1)
            results_f100DP = pd.concat([credit_results_100[f], credit_results_100_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_f100 = pd.concat([results_np, results_f100DP], axis = 1)
            results_f125DP = pd.concat([credit_results_125[f], credit_results_125_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_f125 = pd.concat([results_np, results_f125DP], axis = 1)
            results_f150DP = pd.concat([credit_results_150[f], credit_results_150_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_f150 = pd.concat([results_np, results_f150DP], axis = 1)

        else:
            dataset = "Average"
            if f == "dem-parity":
                results = pd.DataFrame({"mean": [np.mean(avg_dem_real)] + list(np.mean(avg_dem_DP, axis = 1)),
                                        "std": [np.std(avg_dem_real)] + list(np.std(avg_dem_DP, axis = 1))},
                                       index = ["Real", "epsi_1", "epsi_2", "epsi_5", "epsi_8"]).T
                
                results_fdis = pd.DataFrame({"mean": [np.mean(avg_dem_real)] + list(np.mean(avg_dem_fdis, axis = 1)), 
                                             "std": [np.std(avg_dem_real)] + list(np.std(avg_dem_fdis, axis = 1))},
                                            index = ["Real", "epsi_1", "epsi_2", "epsi_5", "epsi_8"]).T

                results_f075 = pd.DataFrame({"mean": [np.mean(avg_dem_real)] + list(np.mean(avg_dem_f075, axis = 1)), 
                                             "std": [np.std(avg_dem_real)] + list(np.std(avg_dem_f075, axis = 1))},
                                            index = ["Real", "epsi_1", "epsi_2", "epsi_5", "epsi_8"]).T
                results_f100 = pd.DataFrame({"mean": [np.mean(avg_dem_real)] + list(np.mean(avg_dem_f100, axis = 1)), 
                                             "std": [np.std(avg_dem_real)] + list(np.std(avg_dem_f100, axis = 1))},
                                            index = ["Real", "epsi_1", "epsi_2", "epsi_5", "epsi_8"]).T
                results_f125 = pd.DataFrame({"mean": [np.mean(avg_dem_real)] + list(np.mean(avg_dem_f125, axis = 1)), 
                                             "std": [np.std(avg_dem_real)] + list(np.std(avg_dem_f125, axis = 1))},
                                            index = ["Real", "epsi_1", "epsi_2", "epsi_5", "epsi_8"]).T
                results_f150 = pd.DataFrame({"mean": [np.mean(avg_dem_real)] + list(np.mean(avg_dem_f150, axis = 1)), 
                                             "std": [np.std(avg_dem_real)] + list(np.std(avg_dem_f150, axis = 1))},
                                            index = ["Real", "epsi_1", "epsi_2", "epsi_5", "epsi_8"]).T

            else:
                results = pd.DataFrame({"mean": [np.mean(avg_dis_real)] + list(np.mean(avg_dis_DP, axis = 1)),
                                        "std": [np.std(avg_dis_real)] + list(np.std(avg_dis_DP, axis = 1))},
                                       index = ["Real", "epsi_1", "epsi_2", "epsi_5", "epsi_8"]).T

                results_fdis = pd.DataFrame({"mean": [np.mean(avg_dis_real)] + list(np.mean(avg_dis_fdis, axis = 1)), 
                                             "std": [np.std(avg_dis_real)] + list(np.std(avg_dis_fdis, axis = 1))},
                                            index = ["Real", "epsi_1", "epsi_2", "epsi_5", "epsi_8"]).T

                results_f075 = pd.DataFrame({"mean": [np.mean(avg_dis_real)] + list(np.mean(avg_dis_f075, axis = 1)), 
                                             "std": [np.std(avg_dis_real)] + list(np.std(avg_dis_f075, axis = 1))},
                                            index = ["Real", "epsi_1", "epsi_2", "epsi_5", "epsi_8"]).T
                results_f100 = pd.DataFrame({"mean": [np.mean(avg_dis_real)] + list(np.mean(avg_dis_f100, axis = 1)), 
                                             "std": [np.std(avg_dis_real)] + list(np.std(avg_dis_f100, axis = 1))},
                                            index = ["Real", "epsi_1", "epsi_2", "epsi_5", "epsi_8"]).T
                results_f125 = pd.DataFrame({"mean": [np.mean(avg_dis_real)] + list(np.mean(avg_dis_f125, axis = 1)), 
                                             "std": [np.std(avg_dis_real)] + list(np.std(avg_dis_f125, axis = 1))},
                                            index = ["Real", "epsi_1", "epsi_2", "epsi_5", "epsi_8"]).T
                results_f150 = pd.DataFrame({"mean": [np.mean(avg_dis_real)] + list(np.mean(avg_dis_f150, axis = 1)), 
                                             "std": [np.std(avg_dis_real)] + list(np.std(avg_dis_f150, axis = 1))},
                                            index = ["Real", "epsi_1", "epsi_2", "epsi_5", "epsi_8"]).T

        results_all = [results, results_fdis, results_f075, results_f100, results_f125, results_f150]
        
        # Customizing the plot.
        col = fairness_clf.index(f)
        ax = axes[c] 
        ax.set_title(f"{dataset} " + r"($\bf{T}$)")
        ax.set_xlabel("$\epsilon$")
        if c == 0:
            if f == "dis-impact":
                ax.set_ylabel(fairness_labels[col])
            else:
                ax.set_ylabel(f"$|${fairness_labels[col]}$|$")
        ax.set_aspect('auto')
        ax.set_xlim([0.6, 8.4])
        ax.set_ylim(limits[f][0], limits[f][1])
        ax.grid(True, linestyle = '--', alpha = 0.6)
    
        # Plotting a dashed line for the demographic parity and disparate impact parameters.
        ax.axhline(0, 0, 5, ls = "--", c = "black")

        if f == "dis-impact":
            # Plotting another dashed line for the disparate impact parameter.
            ax.axhline(0.45, 0, 5, ls = "--", c = "black")
            # Plotting three rectangular patches for the disparate impact parameter.
            rect1 = mpatches.Rectangle(xy = (-0.5, 0), width = 11.5, height = 0.45, color='green', alpha = 0.1, ec='green')
            ax.add_patch(rect1)
            rect2 = mpatches.Rectangle(xy = (-0.5, 0), width = 11.5, height = -10.5, color='red', alpha = 0.1, ec='red')
            ax.add_patch(rect2)
            rect3 = mpatches.Rectangle(xy = (-0.5, 0.45), width = 11.5, height = 10.5, color='red', alpha = 0.1, ec='red')
            ax.add_patch(rect3)

        # Create lineplots for the current metric: Adult, Bank, Credit & Average.
        for res in range(len(results_all)):
            if res == 0:
                ax.plot([0, 11], [results_all[res].iloc[0][0]]*len([0, 11]), color = colors[res])   # original
                ax.plot(epsi, results_all[res].iloc[0][1:5], color = colors[res + 1], marker = markers[res], markersize = msize)   # DP-GAN
            else:
                ax.plot(epsi, results_all[res].iloc[0][1:5], color = colors[res + 1], marker = markers[res], markersize = msize,
                        )   # dataset Fair DP-GANs

        if f == "dem-parity":
            h_patch = [mpatches.Patch(color = colors[i], label = labels[i]) for i in range(1)]
            h_lines = [mlines.Line2D([], [], color = colors[i], marker = markers[i - 1], label = labels[i], linestyle = 'None',
                        markersize = 10) for i in range(1, len(colors))]
            plt.figlegend(handles = h_patch + h_lines, ncol = 7, loc = "upper center", bbox_to_anchor = (0.5, 1.0))
        
        plt.tight_layout(rect = [0, 0, 1, 0.94])
    plt.show()

In [ ]:
# Plotting utility metrics of classifiers trained on the 'best' synthetic datasets from all our models.
labels = ["Original", "DP-GAN", "$Dis$ ($\lambda_f = 0.5$)", "$Dis$ ($\lambda_f = 0.75$)", "$Dis$ ($\lambda_f = 1.0$)", "$Dis$ ($\lambda_f = 1.25$)",
          "$Dis$ ($\lambda_f = 1.5$)"]
colors = ["red", "purple", "darkcyan", "darkorange", "olive", "magenta", "royalblue"]
limits = {"acc": (40, 95), "auroc": (0.485, 0.80)}
utility_clf = ["acc", "auroc"]
utility_labels = ["Accuracy (\%)", "AUROC"]
markers = ["o", "D", "P", "^", "s", "v"]
msize = 9

for u in utility_clf:
    # Create a figure for the aggregated plot.
    fig, axes = plt.subplots(1, 4, figsize = (20, 8.0))  # 1 x 4 grid
    for c in range(0, 4):
        if c == 0:
            dataset = "Adult"
            results_np = pd.concat([adult_results[u]], axis = 1).set_axis(["Real"], axis = 1)
            
            results_DP = pd.concat([adult_results_DP[u], adult_results_DP_std[u]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results = pd.concat([results_np, results_DP], axis = 1)

            results_fdisDP = pd.concat([adult_results_dis[u], adult_results_dis_std[u]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_fdis = pd.concat([results_np, results_fdisDP], axis = 1)

            results_f075DP = pd.concat([adult_results_075[u], adult_results_075_std[u]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_f075 = pd.concat([results_np, results_f075DP], axis = 1)
            results_f100DP = pd.concat([adult_results_100[u], adult_results_100_std[u]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_f100 = pd.concat([results_np, results_f100DP], axis = 1)
            results_f125DP = pd.concat([adult_results_125[u], adult_results_125_std[u]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_f125 = pd.concat([results_np, results_f125DP], axis = 1)
            results_f150DP = pd.concat([adult_results_150[u], adult_results_150_std[u]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_f150 = pd.concat([results_np, results_f150DP], axis = 1)

        elif c == 1:
            dataset = "Bank"
            results_np = pd.concat([bank_results[u]], axis = 1).set_axis(["Real"], axis = 1)
            
            results_DP = pd.concat([bank_results_DP[u], bank_results_DP_std[u]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results = pd.concat([results_np, results_DP], axis = 1)

            results_fdisDP = pd.concat([bank_results_dis[u], bank_results_dis_std[u]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_fdis = pd.concat([results_np, results_fdisDP], axis = 1)

            results_f075DP = pd.concat([bank_results_075[u], bank_results_075_std[u]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_f075 = pd.concat([results_np, results_f075DP], axis = 1)
            results_f100DP = pd.concat([bank_results_100[u], bank_results_100_std[u]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_f100 = pd.concat([results_np, results_f100DP], axis = 1)
            results_f125DP = pd.concat([bank_results_125[u], bank_results_125_std[u]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_f125 = pd.concat([results_np, results_f125DP], axis = 1)
            results_f150DP = pd.concat([bank_results_150[u], bank_results_150_std[u]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_f150 = pd.concat([results_np, results_f150DP], axis = 1)

        elif c == 2:
            dataset = "Credit"
            results_np = pd.concat([credit_results[u]], axis = 1).set_axis(["Real"], axis = 1)
            
            results_DP = pd.concat([credit_results_DP[u], credit_results_DP_std[u]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results = pd.concat([results_np, results_DP], axis = 1)

            results_fdisDP = pd.concat([credit_results_dis[u], credit_results_dis_std[u]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_fdis = pd.concat([results_np, results_fdisDP], axis = 1)

            results_f075DP = pd.concat([credit_results_075[u], credit_results_075_std[u]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_f075 = pd.concat([results_np, results_f075DP], axis = 1)
            results_f100DP = pd.concat([credit_results_100[u], credit_results_100_std[u]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_f100 = pd.concat([results_np, results_f100DP], axis = 1)
            results_f125DP = pd.concat([credit_results_125[u], credit_results_125_std[u]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_f125 = pd.concat([results_np, results_f125DP], axis = 1)
            results_f150DP = pd.concat([credit_results_150[u], credit_results_150_std[u]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_f150 = pd.concat([results_np, results_f150DP], axis = 1)
                
        else:
            dataset = "Average"
            results_np = pd.concat([avg_real_results[u]], axis = 1).set_axis(["Real"], axis = 1)

            results_DP = pd.concat([avg_DP_results[u], avg_DP_results_std[u]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results = pd.concat([results_np, results_DP], axis = 1)

            results_fdisDP = pd.concat([avg_fdis_results[u], avg_fdis_results_std[u]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_fdis = pd.concat([results_np, results_fdisDP], axis = 1)

            results_f075DP = pd.concat([avg_f075_results[u], avg_f075_results_std[u]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_f075 = pd.concat([results_np, results_f075DP], axis = 1)
            results_f100DP = pd.concat([avg_f100_results[u], avg_f100_results_std[u]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_f100 = pd.concat([results_np, results_f100DP], axis = 1)
            results_f125DP = pd.concat([avg_f125_results[u], avg_f125_results_std[u]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_f125 = pd.concat([results_np, results_f125DP], axis = 1)
            results_f150DP = pd.concat([avg_f150_results[u], avg_f150_results_std[u]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_f150 = pd.concat([results_np, results_f150DP], axis = 1)

        results_all = [results, results_fdis, results_f075, results_f100, results_f125, results_f150]
        
        # Customizing the plot.
        col = utility_clf.index(u)
        ax = axes[c] 
        ax.set_title(f"{dataset} " + r"($\bf{T}$)")
        ax.set_xlabel("$\epsilon$")
        if c == 0:
            ax.set_ylabel(utility_labels[col])
        ax.set_aspect('auto')
        ax.set_xlim([0.6, 8.4])
        ax.set_ylim(limits[u][0], limits[u][1])
        ax.grid(True, linestyle = '--', alpha = 0.6)

        # Create lineplots for the current metric: Adult, Bank, Credit & Average.
        for res in range(len(results_all)):
            if res == 0:
                ax.plot([0, 11], [results_all[res].iloc[0][0]]*len([0, 11]), color = colors[res])   # original
                ax.plot(epsi, results_all[res].iloc[0][1:5], color = colors[res + 1], marker = markers[res], markersize = msize)   # DP-GAN
            else:
                ax.plot(epsi, results_all[res].iloc[0][1:5], color = colors[res + 1], marker = markers[res], markersize = msize,
                        )   # dataset Fair DP-GANs

        if u == "acc":
            h_patch = [mpatches.Patch(color = colors[i], label = labels[i]) for i in range(1)]
            h_lines = [mlines.Line2D([], [], color = colors[i], marker = markers[i - 1], label = labels[i], linestyle = 'None',
                        markersize = 10) for i in range(1, len(colors))]
            plt.figlegend(handles = h_patch + h_lines, ncol = 7, loc = "upper center", bbox_to_anchor = (0.5, 1.0))
        
        plt.tight_layout(rect = [0, 0, 1, 0.94])
    plt.show()

In [ ]:
# Plotting fairness metrics of the synthetic datasets from all the GANs.
labels = ["Original", "DP-GAN", "$Dis$ ($\lambda_f = 0.5$)", "$Dis$ ($\lambda_f = 0.75$)", "$Dis$ ($\lambda_f = 1.0$)", "$Dis$ ($\lambda_f = 1.25$)",
          "$Dis$ ($\lambda_f = 1.5$)"]
colors = ["red", "purple", "darkcyan", "darkorange", "olive", "magenta", "royalblue"]
limits = {"dem": (-0.01, 0.32), "dis": (-1.05, 1.05), "eqopp": (-0.01, 0.36)}
fairness_clf = ["dem", "dis", "eqopp"]
fairness_labels = ["Demographic Parity", "Disparate Impact", "Equal Opportunity"]
markers = ["o", "D", "P", "^", "s", "v"]
msize = 9

for f in fairness_clf:
    # Create a figure for the aggregated plot.
    fig, axes = plt.subplots(1, 4, figsize = (20, 8.0))  # 1 x 4 grid
    for c in range(0, 4):
        if c == 0:
            dataset = "Adult"
            results_np = pd.concat([adult_results[f]], axis = 1).set_axis(["Real"], axis = 1)
            
            results_DP = pd.concat([adult_results_DP[f], adult_results_DP_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results = pd.concat([results_np, results_DP], axis = 1)

            results_fdisDP = pd.concat([adult_results_dis[f], adult_results_dis_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_fdis = pd.concat([results_np, results_fdisDP], axis = 1)

            results_f075DP = pd.concat([adult_results_075[f], adult_results_075_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_f075 = pd.concat([results_np, results_f075DP], axis = 1)
            results_f100DP = pd.concat([adult_results_100[f], adult_results_100_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_f100 = pd.concat([results_np, results_f100DP], axis = 1)
            results_f125DP = pd.concat([adult_results_125[f], adult_results_125_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_f125 = pd.concat([results_np, results_f125DP], axis = 1)
            results_f150DP = pd.concat([adult_results_150[f], adult_results_150_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_f150 = pd.concat([results_np, results_f150DP], axis = 1)

        elif c == 1:
            dataset = "Bank"
            results_np = pd.concat([bank_results[f]], axis = 1).set_axis(["Real"], axis = 1)
            
            results_DP = pd.concat([bank_results_DP[f], bank_results_DP_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results = pd.concat([results_np, results_DP], axis = 1)

            results_fdisDP = pd.concat([bank_results_dis[f], bank_results_dis_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_fdis = pd.concat([results_np, results_fdisDP], axis = 1)

            results_f075DP = pd.concat([bank_results_075[f], bank_results_075_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_f075 = pd.concat([results_np, results_f075DP], axis = 1)
            results_f100DP = pd.concat([bank_results_100[f], bank_results_100_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_f100 = pd.concat([results_np, results_f100DP], axis = 1)
            results_f125DP = pd.concat([bank_results_125[f], bank_results_125_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_f125 = pd.concat([results_np, results_f125DP], axis = 1)
            results_f150DP = pd.concat([bank_results_150[f], bank_results_150_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_f150 = pd.concat([results_np, results_f150DP], axis = 1)

        elif c == 2:
            dataset = "Credit"
            results_np = pd.concat([credit_results[f]], axis = 1).set_axis(["Real"], axis = 1)
             
            results_DP = pd.concat([credit_results_DP[f], credit_results_DP_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results = pd.concat([results_np, results_DP], axis = 1)

            results_fdisDP = pd.concat([credit_results_dis[f], credit_results_dis_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_fdis = pd.concat([results_np, results_fdisDP], axis = 1)

            results_f075DP = pd.concat([credit_results_075[f], credit_results_075_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_f075 = pd.concat([results_np, results_f075DP], axis = 1)
            results_f100DP = pd.concat([credit_results_100[f], credit_results_100_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_f100 = pd.concat([results_np, results_f100DP], axis = 1)
            results_f125DP = pd.concat([credit_results_125[f], credit_results_125_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_f125 = pd.concat([results_np, results_f125DP], axis = 1)
            results_f150DP = pd.concat([credit_results_150[f], credit_results_150_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_f150 = pd.concat([results_np, results_f150DP], axis = 1)

        else:
            dataset = "Average"
            results_np = pd.concat([avg_real_results[f]], axis = 1).set_axis(["Real"], axis = 1)

            results_DP = pd.concat([avg_DP_results[f], avg_DP_results_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results = pd.concat([results_np, results_DP], axis = 1)

            results_fdisDP = pd.concat([avg_fdis_results[f], avg_fdis_results_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_fdis = pd.concat([results_np, results_fdisDP], axis = 1)

            results_f075DP = pd.concat([avg_f075_results[f], avg_f075_results_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_f075 = pd.concat([results_np, results_f075DP], axis = 1)
            results_f100DP = pd.concat([avg_f100_results[f], avg_f100_results_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_f100 = pd.concat([results_np, results_f100DP], axis = 1)
            results_f125DP = pd.concat([avg_f125_results[f], avg_f125_results_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_f125 = pd.concat([results_np, results_f125DP], axis = 1)
            results_f150DP = pd.concat([avg_f150_results[f], avg_f150_results_std[f]], axis = 1).set_axis(["mean", "std"], axis = 1).T
            results_f150 = pd.concat([results_np, results_f150DP], axis = 1)

        results_all = [results, results_fdis, results_f075, results_f100, results_f125, results_f150]
        
        # Customizing the plot.
        col = fairness_clf.index(f)
        ax = axes[c] 
        ax.set_title(f"{dataset} " + r"($\bf{T}$)")
        ax.set_xlabel("$\epsilon$")
        if c == 0:
            if f == "dis":
                ax.set_ylabel(fairness_labels[col])
            else:
                ax.set_ylabel(f"$|${fairness_labels[col]}$|$")
        ax.set_aspect('auto')
        ax.set_xlim([0.6, 8.4])
        ax.set_ylim(limits[f][0], limits[f][1])
        ax.grid(True, linestyle = '--', alpha = 0.6)
    
        # Plotting a dashed line for the demographic parity and disparate impact parameters.
        ax.axhline(0, 0, 5, ls = "--", c = "black")

        if f == "dis":
            # Plotting another dashed line for the disparate impact parameter.
            ax.axhline(0.45, 0, 5, ls = "--", c = "black")
            # Plotting three rectangular patches for the disparate impact parameter.
            rect1 = mpatches.Rectangle(xy = (-0.5, 0), width = 11.5, height = 0.45, color='green', alpha = 0.1, ec='green')
            ax.add_patch(rect1)
            rect2 = mpatches.Rectangle(xy = (-0.5, 0), width = 11.5, height = -10.5, color='red', alpha = 0.1, ec='red')
            ax.add_patch(rect2)
            rect3 = mpatches.Rectangle(xy = (-0.5, 0.45), width = 11.5, height = 10.5, color='red', alpha = 0.1, ec='red')
            ax.add_patch(rect3)

        # Create lineplots for the current metric: Adult, Bank, Credit & Average.
        for res in range(len(results_all)):
            if res == 0:
                ax.plot([0, 11], [results_all[res].iloc[0][0]]*len([0, 11]), color = colors[res])   # original
                ax.plot(epsi, results_all[res].iloc[0][1:5], color = colors[res + 1], marker = markers[res], markersize = msize)   # DP-GAN
            else:
                ax.plot(epsi, results_all[res].iloc[0][1:5], color = colors[res + 1], marker = markers[res], markersize = msize,
                            )   # dataset Fair DP-GANs

        if f == "dem":
            h_patch = [mpatches.Patch(color = colors[i], label = labels[i]) for i in range(1)]
            h_lines = [mlines.Line2D([], [], color = colors[i], marker = markers[i - 1], label = labels[i], linestyle = 'None',
                        markersize = 10) for i in range(1, len(colors))]
            plt.figlegend(handles = h_patch + h_lines, ncol = 7, loc = "upper center", bbox_to_anchor = (0.5, 1.0))
        
        plt.tight_layout(rect = [0, 0, 1, 0.94])
    plt.show()

## Comparison of GANs' Results for Group Fairness and Utility

In [ ]:
#adult_results_dis
#bank_results_dis
#credit_results_dis

#adult_results_150
#bank_results_150
#credit_results_150

#adult_results_075
#bank_results_075
#credit_results_075

#adult_results_100
#bank_results_100
#credit_results_100

#adult_results_125
#bank_results_125
#credit_results_125

In [ ]:
#adult_results_DP
#bank_results_DP
#credit_results_DP